## load the verlan pairs(290 pairs in total)

In [3]:
import pandas as pd
df = pd.read_csv("../data/processed/verlan_experiment_core.csv")
# only takes the words included in the wiktionary_verlan_database.csv
# screen = pd.read_csv("../data/processed/wiktionary_verlan_titles.csv")
# title_set = set(screen["title"].dropna().astype(str).str.strip().str.lower())
# df = df[df["verlan_form"].astype(str).str.strip().str.lower().isin(title_set)].copy()

df_two_cols = df[["verlan_form", "base_form"]]

print(df_two_cols.head(100))
print(f"Rows after screen filter: {len(df)}")

   verlan_form base_form
0       barjot    jobard
1       chanmé   méchant
2       chelou    louche
3        cheum     moche
4      chébran   branché
5        duper     perdu
6      foncedé   défoncé
7        kéblo    bloqué
8          ouf       fou
9        relou     lourd
10       reuch      cher
11       scred   discret
12        tebé      bête
13      vénère    énervé
14       zarbi   bizarre
15        ienb      bien
16       cimer     merci
17        zyva     vas-y
18      babtou    toubab
19        beur     arabe
20    caillera  racaille
21      céfran  français
22      genhar    argent
23       iench     chien
24      iencli    client
25       kebla     black
26        keum       mec
27        meuf     femme
28        mifa   famille
29        oinj     joint
30      painco    copain
31      pineco    copine
32       refrè     frère
33       renoi      noir
34        reup      père
35       reuss      sœur
36       skeud    disque
37      streum   monstre
38       tarba    bâtard


## jellyfish

In [4]:
import jellyfish
for i, row in df.iterrows():
    w1 = row["verlan_form"]
    w2 = row["base_form"]
    if not isinstance(w1, str) or not isinstance(w2, str):
        print("not valid strings: ", i, w1, type(w1), w2, type(w2))
lev_dist = []
dam_dist = []
jaro_sim = []
normalize_lev_dist = []
for index, row in df_two_cols.iterrows():
    w1 = row["verlan_form"]
    w2 = row["base_form"]
    lev_dist.append(jellyfish.levenshtein_distance(w1, w2))
    dam_dist.append(jellyfish.damerau_levenshtein_distance(w1, w2))
    jaro_sim.append(jellyfish.jaro_similarity(w1, w2))
    normalize_lev_dist.append(jellyfish.levenshtein_distance(w1,w2)/max(len(w1),len(w2)))
    # soundex1 = jellyfish.soundex(w1)
    # soundex2 = jellyfish.soundex(w2)
    # print(soundex1)
    # print(soundex2)
    
print("Levenshtein distance: ", sum(lev_dist) / len(lev_dist))
print("Normalized Levenshtein distance: ", sum(normalize_lev_dist) / len(normalize_lev_dist))
print("Damerau-Levenshtein distance: ", sum(dam_dist) / len(dam_dist))
print("Jaro similarity: ", sum(jaro_sim) / len(jaro_sim))

Levenshtein distance:  4.339622641509434
Normalized Levenshtein distance:  0.7860512129380054
Damerau-Levenshtein distance:  4.339622641509434
Jaro similarity:  0.38246480982330033


In [12]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# Build a labeled DataFrame
labels = [f"{row['verlan_form']} -> {row['base_form']}" for _, row in df_two_cols.iterrows()]

plot_df = pd.DataFrame({
    "label": labels,
    "verlan": df_two_cols["verlan_form"].values,
    "Levenshtein": lev_dist,
    "Norm. Levenshtein": normalize_lev_dist,
    "Damerau-Levenshtein": dam_dist,
    "Jaro Similarity": jaro_sim,
})

metrics = ["Levenshtein", "Norm. Levenshtein", "Damerau-Levenshtein", "Jaro Similarity"]
palette = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]

# 4 subplots: histogram (frequency) + cumulative line
fig_combo = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=metrics,
    vertical_spacing=0.18,
    horizontal_spacing=0.14,
    specs=[[{"secondary_y": True}, {"secondary_y": True}],
           [{"secondary_y": True}, {"secondary_y": True}]],
 )

for idx, (metric, color) in enumerate(zip(metrics, palette)):
    row, col = divmod(idx, 2)
    row += 1
    col += 1

    values = plot_df[metric].values
    counts, bin_edges = np.histogram(values, bins=20)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    cumulative_counts = np.cumsum(counts)

    # Frequency bars (left y-axis)
    fig_combo.add_trace(
        go.Histogram(
            x=values,
            nbinsx=20,
            marker=dict(color=color, line=dict(color="white", width=0.6)),
            opacity=0.55,
            showlegend=False,
            hovertemplate=f"{metric}<br>Value: %{{x:.4f}}<br>Frequency: %{{y}}<extra></extra>",
        ),
        row=row,
        col=col,
        secondary_y=False,
    )

    # Cumulative frequency line (right y-axis)
    fig_combo.add_trace(
        go.Scatter(
            x=bin_centers,
            y=cumulative_counts,
            mode="lines+markers",
            line=dict(color=color, width=2.4),
            marker=dict(size=5, color=color),
            showlegend=False,
            hovertemplate=f"{metric}<br>Bin center: %{{x:.4f}}<br>Cumulative: %{{y}}<extra></extra>",
        ),
        row=row,
        col=col,
        secondary_y=True,
    )

fig_combo.update_layout(
    title=dict(
        text="Jellyfish Metrics: Frequency + Cumulative Line per Similarity Metric",
        font=dict(size=18),
        x=0.5,
    ),
    barmode="overlay",
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="Arial", size=13),
    height=820,
    margin=dict(t=110, b=50, l=70, r=70),
 )

# Reduce overlap: keep y-axis titles only on outer sides
for r in [1, 2]:
    for c in [1, 2]:
        fig_combo.update_xaxes(showgrid=True, gridcolor="#efefef", row=r, col=c)
        fig_combo.update_yaxes(
            title_text="Frequency" if c == 1 else "",
            showgrid=True,
            gridcolor="#efefef",
            row=r,
            col=c,
            secondary_y=False,
        )
        fig_combo.update_yaxes(
            title_text="Cumulative" if c == 2 else "",
            showgrid=False,
            row=r,
            col=c,
            secondary_y=True,
        )

fig_combo.show()

The distribution of string-based distances shows that French verlan forms are often orthographically far from their standard counterparts. Both raw and normalized Levenshtein distances are generally high, while Jaro similarity is frequently moderate to low. The near-identical distributions of Levenshtein and Damerau-Levenshtein further suggest that verlan is not reducible to simple local character transpositions. Overall, these results indicate that surface-form similarity is insufficient for recovering verlan–base form correspondences, motivating semantic and contextual embedding-based approaches.

## Spacy

In [13]:
import spacy
nlp = spacy.load("fr_core_news_lg")
similarities = []
for index, row in df_two_cols.iterrows():
    w1 = row["verlan_form"]
    w2 = row["base_form"]
    token1 = nlp(w1)
    token2 = nlp(w2)
    sim = token1.similarity(token2)
    similarities.append(sim)
    # print(w1, w2, sim)
print("Average similarity: ", sum(similarities) / len(similarities))
similarities_screen = [sim for sim in similarities if sim > 0]
print("Average similarity (screened): ", sum(similarities_screen) / len(similarities_screen))

/tmp/ipykernel_5943/748199791.py:9: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  sim = token1.similarity(token2)


Average similarity:  0.10221942868369263
Average similarity (screened):  0.2355491182711178


In [14]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Prepare clean pairs
spacy_df = df_two_cols.copy().dropna(subset=["verlan_form", "base_form"]).copy()
spacy_df["verlan_form"] = spacy_df["verlan_form"].astype(str).str.strip()
spacy_df["base_form"] = spacy_df["base_form"].astype(str).str.strip()
spacy_df = spacy_df[(spacy_df["verlan_form"] != "") & (spacy_df["base_form"] != "")].reset_index(drop=True)

# Batch tokenize for efficiency
verlan_docs = list(nlp.pipe(spacy_df["verlan_form"].tolist()))
base_docs = list(nlp.pipe(spacy_df["base_form"].tolist()))

# Keep only pairs with available vectors (especially for verlan words)
spacy_df["verlan_has_vector"] = [doc.vector_norm > 0 for doc in verlan_docs]
spacy_df["base_has_vector"] = [doc.vector_norm > 0 for doc in base_docs]
spacy_df["both_have_vector"] = spacy_df["verlan_has_vector"] & spacy_df["base_has_vector"]

before_n = len(spacy_df)
spacy_df = spacy_df[spacy_df["both_have_vector"]].copy().reset_index(drop=True)
after_n = len(spacy_df)

# Recompute docs after filtering, then compute similarity
verlan_docs = list(nlp.pipe(spacy_df["verlan_form"].tolist()))
base_docs = list(nlp.pipe(spacy_df["base_form"].tolist()))
spacy_df["spacy_similarity"] = [d1.similarity(d2) for d1, d2 in zip(verlan_docs, base_docs)]

avg_all = spacy_df["spacy_similarity"].mean()
avg_nonneg = spacy_df.loc[spacy_df["spacy_similarity"] > 0, "spacy_similarity"].mean()
share_05 = (spacy_df["spacy_similarity"] >= 0.5).mean()
share_07 = (spacy_df["spacy_similarity"] >= 0.7).mean()

print(f"Pairs before vector filter: {before_n}")
print(f"Pairs after vector filter: {after_n}")
print(f"Dropped pairs: {before_n - after_n}")
print(f"Average similarity (all): {avg_all:.4f}")
print(f"Average similarity (>0): {avg_nonneg:.4f}")
print(f"Share(sim >= 0.5): {share_05:.2%}")
print(f"Share(sim >= 0.7): {share_07:.2%}")

# 1) Distribution + key thresholds
fig_hist = px.histogram(
    spacy_df,
    x="spacy_similarity",
    nbins=30,
    marginal="box",
    title="spaCy Similarity Distribution (Vector-Available Pairs)",
    labels={"spacy_similarity": "Cosine similarity"},
)
fig_hist.add_vline(x=avg_all, line_dash="dash", line_color="red", annotation_text=f"mean={avg_all:.2f}")
fig_hist.add_vline(x=0.5, line_dash="dot", line_color="gray", annotation_text="0.5")
fig_hist.add_vline(x=0.7, line_dash="dot", line_color="gray", annotation_text="0.7")
fig_hist.update_layout(template="simple_white")
fig_hist.show()

# 2) Top/Bottom examples for qualitative interpretation
top_n = 15
high_df = spacy_df.nlargest(top_n, "spacy_similarity").copy()
low_df = spacy_df.nsmallest(top_n, "spacy_similarity").copy()
high_df["pair"] = high_df["verlan_form"] + " -> " + high_df["base_form"]
low_df["pair"] = low_df["verlan_form"] + " -> " + low_df["base_form"]

fig_bar = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Top Similar Pairs", "Bottom Similar Pairs"),
    horizontal_spacing=0.12
)
fig_bar.add_trace(
    go.Bar(y=high_df["pair"], x=high_df["spacy_similarity"], orientation="h", name="Top", marker_color="#2E8B57"),
    row=1, col=1
)
fig_bar.add_trace(
    go.Bar(y=low_df["pair"], x=low_df["spacy_similarity"], orientation="h", name="Bottom", marker_color="#B22222"),
    row=1, col=2
)
fig_bar.update_layout(height=620, template="simple_white", showlegend=False, title="Qualitative Cases (Filtered)")
fig_bar.update_xaxes(title_text="Similarity", range=[spacy_df["spacy_similarity"].min() - 0.05, 1.0])
fig_bar.show()

# 3) CDF: easier to read proportion-based conclusions
sorted_sim = np.sort(spacy_df["spacy_similarity"].values)
cdf = np.arange(1, len(sorted_sim) + 1) / len(sorted_sim)
fig_cdf = go.Figure()
fig_cdf.add_trace(go.Scatter(x=sorted_sim, y=cdf, mode="lines", name="CDF"))
fig_cdf.update_layout(
    template="simple_white",
    title="Empirical CDF of spaCy Similarity (Filtered)",
    xaxis_title="Similarity threshold",
    yaxis_title="Proportion of pairs <= threshold",
)
fig_cdf.show()

display(spacy_df.sort_values("spacy_similarity", ascending=False).head(10))

Pairs before vector filter: 53
Pairs after vector filter: 23
Dropped pairs: 30
Average similarity (all): 0.2355
Average similarity (>0): 0.2355
Share(sim >= 0.5): 4.35%
Share(sim >= 0.7): 0.00%


,verlan_form,base_form,verlan_has_vector,base_has_vector,both_have_vector,spacy_similarity
14,keum,mec,True,True,True,0.578722
11,caillera,racaille,True,True,True,0.475571
3,ouf,fou,True,True,True,0.376543
15,meuf,femme,True,True,True,0.367508
1,chelou,louche,True,True,True,0.358081
21,pécho,choper,True,True,True,0.354440
13,kebla,black,True,True,True,0.330492
18,skeud,disque,True,True,True,0.315473
0,chanmé,méchant,True,True,True,0.286485
10,beur,arabe,True,True,True,0.267742


In [16]:
import spacy

nlp = spacy.load("fr_core_news_lg")   # 或 lg
print(nlp.meta["name"])

for w in ["meuf", "femme", "chelou", "louche", "rebeu", "arabe"]:
    doc = nlp(w)
    tok = doc[0]
    print(
        w,
        "has_vector=", tok.has_vector,
        "vector_norm=", tok.vector_norm,
        "is_oov=", tok.is_oov
    )

core_news_lg
meuf has_vector= True vector_norm= 23.746458 is_oov= False
femme has_vector= True vector_norm= 34.48047 is_oov= False
chelou has_vector= True vector_norm= 14.017049 is_oov= False
louche has_vector= True vector_norm= 17.562468 is_oov= False
rebeu has_vector= True vector_norm= 24.750566 is_oov= False
arabe has_vector= True vector_norm= 30.308107 is_oov= False


## Fasttext

In [ ]:
import fasttext
import fasttext.util
ft = fasttext.load_model('../artifacts/models/cc.fr.300.bin')

In [16]:
vec = ft.get_word_vector("meuf")
print(vec.shape)
print(vec[:10])

(300,)
[-0.07452467  0.00135089  0.0236647   0.01450148 -0.07200458 -0.000401
  0.16172828  0.03869662 -0.06638871  0.01467166]


In [17]:
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

w1 = ft.get_word_vector("meuf")
w2 = ft.get_word_vector("femme")

sim = cosine_similarity(w1, w2)
print(sim)
print(ft.get_nearest_neighbors("meuf", k=10))

0.53607875
[(0.7963905930519104, 'nana'), (0.781146228313446, 'gonzesse'), (0.760162889957428, 'pétasse'), (0.7276779413223267, 'mec'), (0.7177290916442871, 'bonnasse'), (0.707476019859314, 'meufs'), (0.7058149576187134, 'pouffiasse'), (0.6935135126113892, 'petasse'), (0.6842332482337952, 'grognasse'), (0.6840963959693909, 'connasse')]


In [18]:
import pandas as pd
import numpy as np

df["fasttext_similarity"] = df.apply(
    lambda row: cosine_similarity(
        ft.get_word_vector(row["verlan_form"]),
        ft.get_word_vector(row["base_form"])
    ),
    axis=1
)
df["fasttext_difference"] = df.apply(
    lambda row: 
        ft.get_word_vector(row["verlan_form"]) - ft.get_word_vector(row["base_form"]),
    axis=1
)



print(df[["verlan_form", "base_form", "fasttext_similarity"]].head(10))
print("Average similarity:", df["fasttext_similarity"].mean())
print("Average difference:", np.linalg.norm(df["fasttext_difference"].mean()))


  verlan_form      base_form  fasttext_similarity
0      babtou         toubab             0.476317
1  balle-peau  peau de balle             0.446452
2      barjot         jobard             0.508903
3       bebar          barbe             0.279475
4       bebon          bombe             0.168011
5      bedave          daube             0.158143
6       beteu           teub             0.074715
7         beu          herbe             0.205081
8      beubar          barbe             0.203733
9      beuher          herbe             0.233043
Average similarity: 0.26055604
Average difference: 0.27838279681174494


In [19]:
import pandas as pd
import numpy as np
from itertools import combinations

def cosine_similarity(a, b):
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na == 0 or nb == 0:
        return np.nan
    return np.dot(a, b) / (na * nb)

# 假设 df 有 verlan_form 和 base_form
df = df.dropna(subset=["verlan_form", "base_form"]).copy()
df["verlan_form"] = df["verlan_form"].astype(str).str.strip()
df["base_form"] = df["base_form"].astype(str).str.strip()

# 只保留单词项，先别混入多词表达
df_single = df[
    ~df["verlan_form"].str.contains(r"\s", regex=True) &
    ~df["base_form"].str.contains(r"\s", regex=True)
].copy()

# 计算 shift 向量
shifts = []
labels = []

for _, row in df_single.iterrows():
    v_verlan = ft.get_word_vector(row["verlan_form"])
    v_base = ft.get_word_vector(row["base_form"])
    shift = v_verlan - v_base
    shifts.append(shift)
    labels.append((row["verlan_form"], row["base_form"]))

shifts = np.array(shifts)

# 1. 看每个 shift 的长度
norms = np.linalg.norm(shifts, axis=1)
print("Average shift norm:", norms.mean())
print("Median shift norm:", np.median(norms))

# 2. 看 shift 两两之间的相似度
pair_sims = []
for i, j in combinations(range(len(shifts)), 2):
    sim = cosine_similarity(shifts[i], shifts[j])
    if not np.isnan(sim):
        pair_sims.append(sim)

print("Average pairwise shift similarity:", np.mean(pair_sims))
print("Median pairwise shift similarity:", np.median(pair_sims))

# 3. 看是否存在整体方向
mean_shift = np.mean(shifts, axis=0)

mean_shift_sims = []
for s in shifts:
    sim = cosine_similarity(s, mean_shift)
    if not np.isnan(sim):
        mean_shift_sims.append(sim)

print("Average similarity to mean shift:", np.mean(mean_shift_sims))
print("Median similarity to mean shift:", np.median(mean_shift_sims))

Average shift norm: 1.2396992
Median shift norm: 1.1618031
Average pairwise shift similarity: 0.05052461
Median pairwise shift similarity: 0.0453516
Average similarity to mean shift: 0.23234175
Median similarity to mean shift: 0.22524567


In [10]:
print(ft.get_nearest_neighbors("meuf", k=10))

[(0.7963905930519104, 'nana'), (0.781146228313446, 'gonzesse'), (0.760162889957428, 'pétasse'), (0.7276779413223267, 'mec'), (0.7177290916442871, 'bonnasse'), (0.707476019859314, 'meufs'), (0.7058149576187134, 'pouffiasse'), (0.6935135126113892, 'petasse'), (0.6842332482337952, 'grognasse'), (0.6840963959693909, 'connasse')]


## Camembert

In [11]:
import re
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

input_file = "../outputs/meuf_sentences.txt"
output_file = "../outputs/meuf_femme_similarity.txt"

# 读取句子
with open(input_file, "r", encoding="utf-8") as f:
    sentences_meuf = [line.strip() for line in f if line.strip()]

pattern_meuf = re.compile(r"\bmeufs?\b", re.IGNORECASE)

def replace_meuf(match):
    word = match.group(0)
    if word.lower() == "meufs":
        return "femmes"
    return "femme"

sentences_femme = [pattern_meuf.sub(replace_meuf, s) for s in sentences_meuf]

# 加载 CamemBERT
tokenizer = AutoTokenizer.from_pretrained("camembert-base")
model = AutoModel.from_pretrained("camembert-base")
model.eval()

def sentence_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
    # 平均池化
    emb = outputs.last_hidden_state.mean(dim=1).squeeze(0)
    return emb

def cosine_similarity(a, b):
    return F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item()

# 计算相似度
results = []
for s_meuf, s_femme in zip(sentences_meuf, sentences_femme):
    emb1 = sentence_embedding(s_meuf)
    emb2 = sentence_embedding(s_femme)
    sim = cosine_similarity(emb1, emb2)
    results.append((s_meuf, s_femme, sim))

# 输出到文件
with open(output_file, "w", encoding="utf-8") as f:
    for s_meuf, s_femme, sim in results:
        f.write(f"ORIGINAL: {s_meuf}\n")
        f.write(f"REPLACED: {s_femme}\n")
        f.write(f"SIMILARITY: {sim:.6f}\n")
        f.write("\n")

# 打印平均值
avg_sim = sum(sim for _, _, sim in results) / len(results)
print("Average similarity:", avg_sim)

/home/wan/miniconda3/envs/llm-verlan/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1435.61it/s]
CamembertModel LOAD REPORT from: camembert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Average similarity: 0.9720009369186208


In [13]:
import re
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import plotly.express as px

# Baseline experiment: compare meuf->femme vs meuf->other words
input_file = "../outputs/meuf_sentences.txt"

if "sentences_meuf" not in globals():
    with open(input_file, "r", encoding="utf-8") as f:
        sentences_meuf = [line.strip() for line in f if line.strip()]

if "tokenizer" not in globals() or "model" not in globals():
    from transformers import AutoTokenizer, AutoModel
    tokenizer = AutoTokenizer.from_pretrained("camembert-base")
    model = AutoModel.from_pretrained("camembert-base")
    model.eval()

def sentence_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).squeeze(0)

def cosine_similarity(a, b):
    return F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item()

pattern_meuf = re.compile(r"\bmeufs?\b", re.IGNORECASE)

def replace_meuf_with_word(text, target_word):
    def _repl(match):
        original = match.group(0)
        is_plural = original.lower().endswith("s")
        return target_word + ("s" if is_plural else "")
    return pattern_meuf.sub(_repl, text)

baseline_words = [
    "femme",      # target pair
    "homme",      # semantically related
    "personne",   # semantically related
    "ami",        # loosely related
    "voiture",    # unrelated
    "pizza",      # unrelated
    "gouvernement",
    "soleil",
]

rows = []
for w in baseline_words:
    replaced_sentences = [replace_meuf_with_word(s, w) for s in sentences_meuf]
    sims = []
    for s_org, s_rep in zip(sentences_meuf, replaced_sentences):
        emb1 = sentence_embedding(s_org)
        emb2 = sentence_embedding(s_rep)
        sims.append(cosine_similarity(emb1, emb2))

    rows.extend([{"target_word": w, "similarity": s} for s in sims])

baseline_df = pd.DataFrame(rows)
summary_df = (
    baseline_df.groupby("target_word", as_index=False)["similarity"]
    .agg(["mean", "std", "median", "min", "max", "count"])
    .reset_index()
    .sort_values("mean", ascending=False)
    .rename(columns={"mean": "mean_similarity", "std": "std_similarity"})
)

print("=== Mean similarity by replacement word ===")
print(summary_df[["target_word", "mean_similarity", "std_similarity", "median", "count"]])

femme_mean = summary_df.loc[summary_df["target_word"] == "femme", "mean_similarity"].iloc[0]
rank = int((summary_df["mean_similarity"] > femme_mean).sum() + 1)
print(f"\nmeuf -> femme mean similarity: {femme_mean:.4f} (rank {rank}/{len(summary_df)})")

fig = px.box(
    baseline_df,
    x="target_word",
    y="similarity",
    points="outliers",
    title="Baseline: Similarity of 'meuf' replaced by different words",
    labels={"target_word": "Replacement word", "similarity": "Sentence cosine similarity"},
)
fig.update_layout(template="simple_white")
fig.show()

fig_mean = px.bar(
    summary_df,
    x="target_word",
    y="mean_similarity",
    error_y="std_similarity",
    title="Mean Similarity with Std (Baseline Comparison)",
    labels={"target_word": "Replacement word", "mean_similarity": "Mean sentence similarity"},
)
fig_mean.update_layout(template="simple_white")
fig_mean.show()

=== Mean similarity by replacement word ===
    target_word  mean_similarity  std_similarity    median  count
1         femme         0.972001        0.021711  0.977730     79
7       voiture         0.963408        0.022146  0.968742     79
4      personne         0.959512        0.025177  0.964563     79
5         pizza         0.956271        0.024021  0.958895     79
0           ami         0.947314        0.035974  0.953067     79
3         homme         0.944836        0.043104  0.954562     79
6        soleil         0.933911        0.040549  0.940845     79
2  gouvernement         0.932491        0.040780  0.938191     79

meuf -> femme mean similarity: 0.9720 (rank 1/8)


In [12]:
import re
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

input_file = "../outputs/meuf_sentences.txt"

# 读取原句
with open(input_file, "r", encoding="utf-8") as f:
    sentences_meuf = [line.strip() for line in f if line.strip()]

pattern_meuf = re.compile(r"\bmeufs?\b", re.IGNORECASE)

def replace_meuf(match):
    word = match.group(0)
    if word.lower() == "meufs":
        return "femmes"
    return "femme"

sentences_femme = [pattern_meuf.sub(replace_meuf, s) for s in sentences_meuf]

# 加载 CamemBERT
tokenizer = AutoTokenizer.from_pretrained("camembert-base", use_fast=True)
model = AutoModel.from_pretrained("camembert-base")
model.eval()

def find_word_span(text, word):
    """
    返回目标词在字符串中的字符区间 [start, end)
    只取第一个完整匹配
    """
    pattern = re.compile(rf"\b{re.escape(word)}\b", re.IGNORECASE)
    m = pattern.search(text)
    if m is None:
        return None
    return m.start(), m.end()

def get_target_embedding(text, target_word):
    """
    提取 target_word 在 text 里的 contextual embedding
    做法：找到覆盖该词的所有 subword token，平均它们的向量
    """
    span = find_word_span(text, target_word)
    if span is None:
        return None

    char_start, char_end = span

    encoded = tokenizer(
        text,
        return_tensors="pt",
        return_offsets_mapping=True,
        truncation=True,
        max_length=128
    )

    offset_mapping = encoded.pop("offset_mapping")[0].tolist()

    with torch.no_grad():
        outputs = model(**encoded)

    # shape: [seq_len, hidden_size]
    hidden = outputs.last_hidden_state[0]

    token_indices = []
    for i, (tok_start, tok_end) in enumerate(offset_mapping):
        # 跳过 special tokens，它们通常 offset 为 (0, 0)
        if tok_start == tok_end == 0:
            continue

        # 只要 token 和目标词字符区间有重叠，就算进去
        if not (tok_end <= char_start or tok_start >= char_end):
            token_indices.append(i)

    if not token_indices:
        return None

    target_vec = hidden[token_indices].mean(dim=0)
    return target_vec

def cosine_similarity(a, b):
    return F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item()

results = []

for s_meuf, s_femme in zip(sentences_meuf, sentences_femme):
    vec_meuf = get_target_embedding(s_meuf, "meuf")
    vec_femme = get_target_embedding(s_femme, "femme")

    if vec_meuf is None or vec_femme is None:
        continue

    sim = cosine_similarity(vec_meuf, vec_femme)
    results.append((s_meuf, s_femme, sim))

# 输出结果
for s_meuf, s_femme, sim in results[:10]:
    print("ORIGINAL:", s_meuf)
    print("REPLACED:", s_femme)
    print("TARGET WORD SIMILARITY:", round(sim, 6))
    print()

avg_sim = sum(x[2] for x in results) / len(results)
print("Average target-word similarity:", avg_sim)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2013.54it/s]
CamembertModel LOAD REPORT from: camembert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ORIGINAL: Ça, c'est de la meuf!
REPLACED: Ça, c'est de la femme!
TARGET WORD SIMILARITY: 0.615303

ORIGINAL: Tu vois... j'ai sauvé son petit cul... et c'est lui qui a emballé la meuf.
REPLACED: Tu vois... j'ai sauvé son petit cul... et c'est lui qui a emballé la femme.
TARGET WORD SIMILARITY: 0.62326

ORIGINAL: Ouais, c'est un peu comme être avec une meuf sans dents.
REPLACED: Ouais, c'est un peu comme être avec une femme sans dents.
TARGET WORD SIMILARITY: 0.642098

ORIGINAL: Sans meuf, on baise pas.
REPLACED: Sans femme, on baise pas.
TARGET WORD SIMILARITY: 0.58248

ORIGINAL: La meuf a reçu un verre.
REPLACED: La femme a reçu un verre.
TARGET WORD SIMILARITY: 0.658817

ORIGINAL: Cette meuf te kife, mec.
REPLACED: Cette femme te kife, mec.
TARGET WORD SIMILARITY: 0.664354

ORIGINAL: Je vais me taper une meuf avant le mariage.
REPLACED: Je vais me taper une femme avant le mariage.
TARGET WORD SIMILARITY: 0.695051

ORIGINAL: Cette meuf, Stephanie, y a rien à jeter chez elle.
REPLACED: 

In [14]:
import re
import numpy as np
import pandas as pd
import plotly.express as px

# Target-word baseline after the previous block
if "sentences_meuf" not in globals():
    input_file = "../outputs/meuf_sentences.txt"
    with open(input_file, "r", encoding="utf-8") as f:
        sentences_meuf = [line.strip() for line in f if line.strip()]

if "get_target_embedding" not in globals():
    def find_word_span(text, word):
        pattern = re.compile(rf"\\b{re.escape(word)}\\b", re.IGNORECASE)
        m = pattern.search(text)
        if m is None:
            return None
        return m.start(), m.end()

    def get_target_embedding(text, target_word):
        span = find_word_span(text, target_word)
        if span is None:
            return None

        char_start, char_end = span
        encoded = tokenizer(
            text,
            return_tensors="pt",
            return_offsets_mapping=True,
            truncation=True,
            max_length=128,
        )
        offset_mapping = encoded.pop("offset_mapping")[0].tolist()

        with torch.no_grad():
            outputs = model(**encoded)
        hidden = outputs.last_hidden_state[0]

        token_indices = []
        for i, (tok_start, tok_end) in enumerate(offset_mapping):
            if tok_start == tok_end == 0:
                continue
            if not (tok_end <= char_start or tok_start >= char_end):
                token_indices.append(i)

        if not token_indices:
            return None
        return hidden[token_indices].mean(dim=0)

if "cosine_similarity" not in globals():
    def cosine_similarity(a, b):
        return F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item()

pattern_meuf_or_plural = re.compile(r"\bmeufs?\b", re.IGNORECASE)

def replace_meuf_with_word_singular(text, target_word):
    # Always replace by singular form to make target extraction stable.
    return pattern_meuf_or_plural.sub(target_word, text)

def get_meuf_embedding_from_original(text):
    # Try both meuf and meufs in original sentence.
    vec = get_target_embedding(text, "meuf")
    if vec is None:
        vec = get_target_embedding(text, "meufs")
    return vec

baseline_words_target = [
    "femme",
    "homme",
    "personne",
    "ami",
    "voiture",
    "pizza",
    "gouvernement",
    "soleil",
]

rows = []
for w in baseline_words_target:
    kept = 0
    sims = []
    for s_org in sentences_meuf:
        s_rep = replace_meuf_with_word_singular(s_org, w)
        vec_org = get_meuf_embedding_from_original(s_org)
        vec_rep = get_target_embedding(s_rep, w)

        if vec_org is None or vec_rep is None:
            continue

        sim = cosine_similarity(vec_org, vec_rep)
        sims.append(sim)
        kept += 1

    rows.extend([{"target_word": w, "similarity": s} for s in sims])
    print(f"{w}: valid pairs = {kept}, mean = {np.mean(sims):.4f}" if kept > 0 else f"{w}: valid pairs = 0")

target_baseline_df = pd.DataFrame(rows)
target_summary_df = (
    target_baseline_df.groupby("target_word", as_index=False)["similarity"]
    .agg(["mean", "std", "median", "count", "min", "max"])
)
target_summary_df.columns = [
    "target_word", "mean_similarity", "std_similarity", "median", "count", "min", "max"
]
target_summary_df = target_summary_df.sort_values("mean_similarity", ascending=False).reset_index(drop=True)

print("\n=== Target-word baseline summary ===")
print(target_summary_df[["target_word", "mean_similarity", "std_similarity", "median", "count"]])

femme_mean = target_summary_df.loc[target_summary_df["target_word"] == "femme", "mean_similarity"].iloc[0]
femme_rank = int((target_summary_df["mean_similarity"] > femme_mean).sum() + 1)
print(f"\nTarget-word level: meuf -> femme mean similarity = {femme_mean:.4f} (rank {femme_rank}/{len(target_summary_df)})")

fig_box_target = px.box(
    target_baseline_df,
    x="target_word",
    y="similarity",
    points="outliers",
    title="Target-Word Baseline: meuf vs replacement-word contextual embedding",
    labels={"target_word": "Replacement word", "similarity": "Target-word cosine similarity"},
)
fig_box_target.update_layout(template="simple_white")
fig_box_target.show()

fig_bar_target = px.bar(
    target_summary_df,
    x="target_word",
    y="mean_similarity",
    error_y="std_similarity",
    title="Target-Word Baseline Mean Similarity (with Std)",
    labels={"target_word": "Replacement word", "mean_similarity": "Mean target-word similarity"},
)
fig_bar_target.update_layout(template="simple_white")
fig_bar_target.show()

femme: valid pairs = 79, mean = 0.6131
homme: valid pairs = 79, mean = 0.5047
personne: valid pairs = 79, mean = 0.5139
ami: valid pairs = 79, mean = 0.5025
voiture: valid pairs = 79, mean = 0.5580
pizza: valid pairs = 79, mean = 0.4438
gouvernement: valid pairs = 79, mean = 0.4231
soleil: valid pairs = 79, mean = 0.3939

=== Target-word baseline summary ===
    target_word  mean_similarity  std_similarity    median  count
0         femme         0.613097        0.089797  0.621391     79
1       voiture         0.558017        0.109033  0.562692     79
2      personne         0.513909        0.104595  0.494961     79
3         homme         0.504705        0.128461  0.517643     79
4           ami         0.502518        0.116018  0.505474     79
5         pizza         0.443842        0.106193  0.438345     79
6  gouvernement         0.423142        0.144220  0.440241     79
7        soleil         0.393929        0.130639  0.399577     79

Target-word level: meuf -> femme mean simila